# 01 — Data Collection

This notebook pulls raw shot log data from the NBA API for individual players.
A shot log is a record of every field goal attempt in a season — location on the court, whether it went in, shot type, quarter, defender distance, and more.

We'll save every API response to a local cache folder so we never have to re-download the same data twice.

**Section 1 covers:**
- Setting up our imports
- Building a reusable fetch function that checks the cache before hitting the API
- Testing it with LeBron James (2023–24 regular season)

## 1.1 — Imports

We need a handful of standard libraries plus `nba_api`, the community-built Python wrapper around NBA.com's stats endpoints.

- `os` / `pathlib` — create folders and check if files exist on disk
- `time` / `random` — controlled delays between API calls to avoid rate-limiting; `random` lets us vary the delay so requests don't arrive at perfectly regular intervals (which triggers some firewalls)
- `pandas` — store and manipulate the shot data as a DataFrame
- `requests.exceptions` — lets us catch specific network errors (timeouts, dropped connections) for the retry logic we'll build into the fetch function
- `nba_api` — `ShotChartDetail` for shot-by-shot data; `CommonAllPlayers` for the full league roster; `players` (static) for quick name-to-ID lookups

In [1]:
import os
import time
import random
import pandas as pd

from pathlib import Path
from requests.exceptions import ReadTimeout, ConnectionError as ConnError
from nba_api.stats.endpoints import ShotChartDetail, CommonAllPlayers
from nba_api.stats.static import players

print("Imports loaded successfully.")
print(f"pandas version: {pd.__version__}")

Imports loaded successfully.
pandas version: 3.0.3


## 1.2 — Directory Setup

We'll store cached API responses in `data/cache/`. Each file is named after the player ID and season so it's easy to find later (e.g., `shots_2544_2023-24.csv`).

This cell just ensures the folder exists before we try to write into it.

In [2]:
# Resolve paths relative to the notebook's location so this works
# regardless of where Jupyter was launched from.
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent  # one level up from /notebooks/

CACHE_DIR = PROJECT_ROOT / "data" / "cache"
RAW_DIR   = PROJECT_ROOT / "data" / "raw"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Cache folder : {CACHE_DIR}")
print(f"Raw folder   : {RAW_DIR}")

Project root : /Users/narayanlekhi/Documents/GitHub/nba-shot-analytics
Cache folder : /Users/narayanlekhi/Documents/GitHub/nba-shot-analytics/data/cache
Raw folder   : /Users/narayanlekhi/Documents/GitHub/nba-shot-analytics/data/raw


## 1.3 — Player ID Lookup

The NBA API identifies every player by a numeric ID, not a name. LeBron James's ID is `2544`.

The `nba_api.stats.static.players` module ships with a full player list built in — no network request needed. We'll write a small helper that takes a name and returns the ID, so we don't have to memorize numbers.

In [3]:
def get_player_id(full_name: str) -> int:
    """
    Look up an NBA player's numeric ID by their full name.
    Raises a clear error if the name isn't found, rather than returning None silently.
    """
    matches = players.find_players_by_full_name(full_name)

    if not matches:
        raise ValueError(
            f"No player found for '{full_name}'. "
            "Check spelling — the API expects the full legal name (e.g. 'LeBron James')."
        )

    # find_players_by_full_name can return multiple partial matches;
    # take the first exact-ish match (results are sorted by relevance).
    player = matches[0]
    print(f"Found: {player['full_name']}  |  ID: {player['id']}  |  Active: {player['is_active']}")
    return player["id"]


# Test it
lebron_id = get_player_id("LeBron James")

Found: LeBron James  |  ID: 2544  |  Active: True


## 1.4 — Fetch Function with Caching and Retry Logic

This is the core fetch function. It now does three things:

1. **Check the cache first.** If the player's shot log for this season is already on disk, load it immediately — no API call, no delay.
2. **Randomized delay before fresh requests.** A random 3–5 second pause before each new API call avoids hitting NBA.com's rate limiter, which is more likely to block evenly-spaced automated requests than irregular ones.
3. **Retry on transient network errors.** If the request fails with a timeout or dropped connection, wait 5 seconds and try again, up to 3 times. On a final failure it returns `None` so the calling loop can log it and continue rather than crashing.

The function returns `None` (rather than raising an exception) so the multi-player loop in Section 2 can catch failures gracefully and move on.

In [4]:
def fetch_shot_log(
    player_id: int,
    season: str,
    cache_dir: Path = None,
    max_retries: int = 3,
) -> pd.DataFrame | None:
    """
    Return a DataFrame of every field goal attempt for a player in a given season.

    Checks data/cache/ first. On a cache miss, waits a random 3–5 seconds then
    calls the NBA Aerrors.
    Returns None if all retries are exhausted.

    Args:
        player_id:   NBA player ID (e.g. 2544 for LeBron James)
        season:      Season string in "YYYY-YY" format (e.g. "2025-26")
        cache_dir:   Folder for cached CSVs; defaults to CACHE_DIR
        max_retries: How many times to retry on a timeout or connection error

    Returns:
        pandas DataFrame with one row per shot attempt, or None on failure
    """
    if cache_dir is None:
        cache_dir = CACHE_DIR

    cache_file = cache_dir / f"shots_{player_id}_{season}.csv"

    # --- Cache hit: load from disk, no API call needed ---
    if cache_file.exists():
        return pd.read_csv(cache_file)

    # --- Cache miss: randomized delay then API call ---
    delay = random.uniform(3.0, 5.0)
    time.sleep(delay)

    for attempt in range(1, max_retries + 1):
        try:
            shot_chart = ShotChartDetail(
                team_id=0,
                player_id=player_id,
                season_nullable=season,
                season_type_all_star="Regular Season",
                context_measure_simple="FGA",
            )
            df = shot_chart.get_data_frames()[0]
            df.to_csv(cache_file, index=False)
            return df

        except (ReadTimeout, ConnError) as e:
            # Transient network error — eligible for retry
            if attempt < max_retries:
                print(f"      [retry {attempt}/{max_retries}] {type(e).__name__} — waiting 5s...")
                time.sleep(5.0)
            else:
                print(f"      [failed] {type(e).__name__} after {max_retries} attempts")
                return None

        except Exception as e:
            # Unexpected error — don't retry, just log and bail
            print(f"      [error] {type(e).__name__}: {e}")
            return None

## 1.5 — Test: LeBron James, 2025–26 Season

We call the function once before the full-league loop to confirm it works correctly under the new season and retry logic. If LeBron's 2025–26 cache file doesn't exist yet, this will make one API call (with the 3–5 second delay). If it does exist, it loads instantly from disk.

In [5]:
SEASON = "2025-26"

lebron_id    = get_player_id("LeBron James")
lebron_shots = fetch_shot_log(player_id=lebron_id, season=SEASON)

if lebron_shots is None:
    print("Fetch failed — check your network connection and re-run.")
else:
    print(f"\nShape: {lebron_shots.shape[0]:,} rows × {lebron_shots.shape[1]} columns")
    print("\nColumns:")
    print(lebron_shots.columns.tolist())

Found: LeBron James  |  ID: 2544  |  Active: True

Shape: 919 rows × 24 columns

Columns:
['GRID_TYPE', 'GAME_ID', 'GAME_EVENT_ID', 'PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_NAME', 'PERIOD', 'MINUTES_REMAINING', 'SECONDS_REMAINING', 'EVENT_TYPE', 'ACTION_TYPE', 'SHOT_TYPE', 'SHOT_ZONE_BASIC', 'SHOT_ZONE_AREA', 'SHOT_ZONE_RANGE', 'SHOT_DISTANCE', 'LOC_X', 'LOC_Y', 'SHOT_ATTEMPTED_FLAG', 'SHOT_MADE_FLAG', 'GAME_DATE', 'HTM', 'VTM']


## 1.6 — Quick Sanity Check

Before we close out Section 1, let's verify a few things:
- The data is actually LeBron's (check `PLAYER_NAME`)
- The season looks right (check `SEASON_ID`)
- We have a mix of made and missed shots (not all zeros in `SHOT_MADE_FLAG`)

In [6]:
# Spot-check: who and what season is this?
print("Player name (unique values):", lebron_shots["PLAYER_NAME"].unique())

# SEASON_ID is not returned by ShotChartDetail; use GAME_DATE instead
print("Date range:", lebron_shots["GAME_DATE"].min(), "→", lebron_shots["GAME_DATE"].max())

# Counts of made vs missed shots
shot_counts = lebron_shots["SHOT_MADE_FLAG"].value_counts().rename({0: "Missed", 1: "Made"})
print("\nShot outcome counts:")
print(shot_counts)
print(f"\nImplied FG%: {shot_counts['Made'] / shot_counts.sum():.1%}")

Player name (unique values): <StringArray>
['LeBron James']
Length: 1, dtype: str
Date range: 20251118 → 20260412

Shot outcome counts:
SHOT_MADE_FLAG
Made      473
Missed    446
Name: count, dtype: int64

Implied FG%: 51.5%


## Section 2 — Fetching the Full 2025–26 League Roster

Instead of a hand-picked list of five players, we now pull shot logs for every player who appeared in the 2025–26 regular season.

The plan:
1. Fetch the full active roster from the NBA API's `CommonAllPlayers` endpoint
2. Loop through every player, using the cache to skip anyone already downloaded
3. Retry up to 3 times on network errors; log any players who ultimately fail
4. Combine all successful results into one DataFrame and save to `data/raw/shots_2025_26.csv`

**Estimated runtime:** With ~400–500 active players and a 3–5 second delay per fresh request, expect 30–60 minutes on a first run. Subsequent runs (all cache hits) complete in under a minute. If the script is interrupted, just re-run — the cache preserves every completed player automatically.

### 2.1 — Fetch the Active Player Roster

`CommonAllPlayers` returns a table of every player in NBA history. Passing `is_only_current_season=1` restricts it to the season we specify. We then filter to players where `GAMES_PLAYED_FLAG == "Y"` — this drops players who were on a roster but never appeared in a game and would return an empty shot log anyway.

We print a count so we know how many API calls the fetch loop will need to make on a cold run.

In [7]:
print(f"Fetching active roster for {SEASON}...")

roster_response = CommonAllPlayers(
    is_only_current_season=1,
    league_id="00",
    season=SEASON,
)
all_players_df = roster_response.get_data_frames()[0]

# Keep only players who actually appeared in at least one game
active_players_df = all_players_df[all_players_df["GAMES_PLAYED_FLAG"] == "Y"].copy()
active_players_df = active_players_df.reset_index(drop=True)

print(f"\nTotal players in API response : {len(all_players_df):,}")
print(f"Players with games played      : {len(active_players_df):,}")
print(f"\nFirst 10 players:")
display(
    active_players_df[["PERSON_ID", "DISPLAY_FIRST_LAST", "TEAM_ABBREVIATION"]]
    .head(10)
)

Fetching active roster for 2025-26...

Total players in API response : 582
Players with games played      : 582

First 10 players:


,PERSON_ID,DISPLAY_FIRST_LAST,TEAM_ABBREVIATION
0,1630173,Precious Achiuwa,SAC
1,203500,Steven Adams,HOU
2,1628389,Bam Adebayo,MIA
3,1630534,Ochai Agbaji,BKN
4,1630583,Santi Aldama,MEM
5,1641725,Trey Alexander,NOP
6,1629638,Nickeil Alexander-Walker,ATL
7,1628960,Grayson Allen,PHX
8,1628386,Jarrett Allen,CLE
9,1630631,Jose Alvarado,NYK


### 2.2 — Fetch Shot Logs for All Players

The loop iterates over every player in the roster. For each one:

- **Cache hit** → load silently from disk, no output, no delay
- **Cache miss** → call `fetch_shot_log()`, which applies the 3–5 second randomized delay and up to 3 retries automatically
- **Empty result** → player had 0 field goal attempts (rare: some two-way or end-of-bench players), logged separately and skipped
- **None result** → all retries failed, player added to `failed_players` list

The `tqdm` progress bar shows overall progress. Intermediate `tqdm.write()` calls print status lines without corrupting the bar.

If you need to stop mid-run, `Ctrl+C` is safe — every player fetched so far is already saved to the cache. Re-running the cell will skip all cached players and resume from where it stopped.

In [8]:
from tqdm import tqdm

all_shot_logs   = []   # one DataFrame per player (non-empty)
failed_players  = []   # players where all retries failed
skipped_players = []   # players with 0 shot attempts
cache_hits      = 0

for _, player_row in tqdm(
    active_players_df.iterrows(),
    total=len(active_players_df),
    desc="Fetching shot logs",
):
    name      = player_row["DISPLAY_FIRST_LAST"]
    player_id = int(player_row["PERSON_ID"])

    cache_file = CACHE_DIR / f"shots_{player_id}_{SEASON}.csv"

    # --- Fast path: cache hit ---
    # Read silently and move on — no print, no delay.
    if cache_file.exists():
        cache_hits += 1
        df = pd.read_csv(cache_file)
        if len(df) > 0:
            df["NAME"] = name
            all_shot_logs.append(df)
        continue

    # --- Slow path: API call (delay + retry handled inside fetch_shot_log) ---
    df = fetch_shot_log(player_id=player_id, season=SEASON)

    if df is None:
        failed_players.append({"name": name, "id": player_id})
        tqdm.write(f"  [FAILED]  {name}")

    elif len(df) == 0:
        skipped_players.append(name)
        tqdm.write(f"  [skip]    {name} — 0 shot attempts recorded")

    else:
        df["NAME"] = name
        all_shot_logs.append(df)
        tqdm.write(f"  [done]    {name} — {len(df):,} shots")

print(f"\nLoop complete.")
print(f"  Loaded from cache : {cache_hits}")
print(f"  Fetched via API   : {len(all_shot_logs) + len(failed_players) + len(skipped_players) - cache_hits}")
print(f"  Successful        : {len(all_shot_logs)}")
print(f"  Skipped (0 shots) : {len(skipped_players)}")
print(f"  Failed            : {len(failed_players)}")

Fetching shot logs:   0%|                      | 1/582 [00:05<56:08,  5.80s/it]

  [done]    Precious Achiuwa — 598 shots


Fetching shot logs:   0%|                      | 2/582 [00:10<50:59,  5.27s/it]

  [done]    Steven Adams — 139 shots


Fetching shot logs:   1%|                      | 3/582 [00:15<49:11,  5.10s/it]

...

  [done]    Rocco Zikarsky — 8 shots


Fetching shot logs: 100%|███████████████████▉| 581/582 [45:29<00:04,  4.69s/it]

  [done]    Ivica Zubac — 482 shots


Fetching shot logs: 100%|████████████████████| 582/582 [45:35<00:00,  4.70s/it]

  [done]    Tristan da Silva — 620 shots

Loop complete.
  Loaded from cache : 1
  Fetched via API   : 581
  Successful        : 582
  Skipped (0 shots) : 0
  Failed            : 0


### 2.3 — Retry Failed Players

If any players ended up in `failed_players`, we make one more pass at them here. A player who failed during the main loop likely hit a brief NBA.com outage — a fresh attempt a few minutes later usually succeeds. We use a longer fixed delay (8 seconds) for these retries to give the API more breathing room.

If `failed_players` is empty, this cell does nothing and can be skipped.

In [9]:
if not failed_players:
    print("No failed players — nothing to retry.")
else:
    print(f"Retrying {len(failed_players)} failed player(s)...\n")
    still_failed = []

    for entry in failed_players:
        name, player_id = entry["name"], entry["id"]
        print(f"  Retrying {name}...")
        time.sleep(8.0)

        df = fetch_shot_log(player_id=player_id, season=SEASON)

        if df is None:
            still_failed.append(entry)
            print(f"    [FAILED again] {name}")
        elif len(df) == 0:
            print(f"    [skip] {name} — 0 shots")
        else:
            df["NAME"] = name
            all_shot_logs.append(df)
            print(f"    [recovered] {name} — {len(df):,} shots")

    failed_players = still_failed
    print(f"\nAfter retry — still failed: {len(failed_players)}")
    if failed_players:
        print("Permanently failed players:")
        for entry in failed_players:
            print(f"  {entry['name']} (ID: {entry['id']})")

No failed players — nothing to retry.


### 2.4 — Combine and Save to `data/raw/`

We stack all per-player DataFrames into one combined DataFrame and save it to `data/raw/shots_2025_26.csv`. This is the raw source of truth for the rest of the project — notebook 02 reads from here and all cleaning happens downstream.

If `all_shot_logs` is empty (every player failed), we print a warning and stop rather than writing an empty file.

In [10]:
if not all_shot_logs:
    print("ERROR: all_shot_logs is empty — no data to save. Check failed_players and retry.")
else:
    combined_shots = pd.concat(all_shot_logs, ignore_index=True)

    raw_output_path = RAW_DIR / "shots_2025_26.csv"
    combined_shots.to_csv(raw_output_path, index=False)

    print(f"Saved  : {raw_output_path}")
    print(f"Rows   : {len(combined_shots):,}")
    print(f"Cols   : {len(combined_shots.columns)}")
    print(f"Players: {combined_shots['NAME'].nunique():,}")

Saved  : /Users/narayanlekhi/Documents/GitHub/nba-shot-analytics/data/raw/shots_2025_26.csv
Rows   : 219,160
Cols   : 25
Players: 582


### 2.5 — Final Summary

A quick verification that the combined dataset looks reasonable: player count, shot count distribution, and a top-10 by volume. We also print the failed player list one final time so it's visible in the notebook's output history.

In [11]:
shots_per_player = (
    combined_shots.groupby("NAME")
    .size()
    .rename("FGA")
    .sort_values(ascending=False)
)

print(f"=== 2025-26 Shot Log Summary ===")
print(f"Players      : {len(shots_per_player):,}")
print(f"Total FGA    : {shots_per_player.sum():,}")
print(f"Median FGA   : {shots_per_player.median():.0f} per player")
print(f"Min FGA      : {shots_per_player.min()} ({shots_per_player.idxmin()})")
print(f"Max FGA      : {shots_per_player.max()} ({shots_per_player.idxmax()})")

print(f"\nTop 10 by shot volume:")
display(shots_per_player.head(10).reset_index())

print(f"\nFailed players (excluded from CSV): {len(failed_players)}")
for entry in failed_players:
    print(f"  {entry['name']} (ID: {entry['id']})")
if not failed_players:
    print("  None — full dataset is complete.")

=== 2025-26 Shot Log Summary ===
Players      : 582
Total FGA    : 219,160
Median FGA   : 292 per player
Min FGA      : 1 (Tosan Evbuomwan)
Max FGA      : 1543 (Jaylen Brown)

Top 10 by shot volume:


,NAME,FGA
0,Jaylen Brown,1543
1,Tyrese Maxey,1501
2,Jalen Brunson,1475
3,Luka Dončić,1457
4,Donovan Mitchell,1403
5,Kevin Durant,1376
6,Jamal Murray,1360
7,Shai Gilgeous-Alexander,1321
8,Brandon Ingram,1288
9,Kawhi Leonard,1258



Failed players (excluded from CSV): 0
  None — full dataset is complete.


## Section 3 — Pivot Verification: Contest Data and Qualifying Player Pool

Before building the new PPS-based analysis, we need to confirm two things directly from the raw data:

1. **`CLOSE_DEF_DIST`** — does this column exist in `shots_2025_26.csv`? If yes, how complete is it and what does the distribution look like so we can validate that the contest buckets (0–2 ft, 2–4 ft, 4–6 ft, 6+ ft) are meaningful?
2. **300-shot minimum filter** — how many players in the 2025–26 raw data have at least 300 field goal attempts, and how many fall below that threshold?

Both checks must be confirmed here before any downstream notebooks are updated.

In [12]:
import pandas as pd
from pathlib import Path

# Re-declare paths so this works after a kernel restart
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_FILE     = PROJECT_ROOT / "data" / "raw" / "shots_2025_26.csv"

raw = pd.read_csv(RAW_FILE)

# ── Check 1: Does CLOSE_DEF_DIST exist in the raw data? ──────────────────
print("=" * 60)
print("CHECK 1 — CLOSE_DEF_DIST column")
print("=" * 60)

DIST_COL = "CLOSE_DEF_DIST"

if DIST_COL not in raw.columns:
    # Column is missing — explain why and what it would take to get it.
    # ShotChartDetail with context_measure_simple='FGA' returns shot location
    # and outcome only. To get defender distance we need 'DFGA' (defended FGA),
    # which returns a different result set including CLOSE_DEF_DIST.
    print(f"\n  COLUMN NOT FOUND: '{DIST_COL}' is not in shots_2025_26.csv")
    print(f"\n  The current raw file has {len(raw.columns)} columns:")
    for col in raw.columns:
        print(f"    {col}")
    print()
    print("  WHY IT IS MISSING:")
    print("  Notebook 01 fetches with context_measure_simple='FGA'.")
    print("  That returns shot coordinates + outcome, but NOT defender data.")
    print("  To get CLOSE_DEF_DIST, the fetch must use context_measure_simple='DFGA'.")
    print("  The raw data needs to be re-collected before the contest-level")
    print("  breakdown can be implemented.")

else:
    # Column exists — print completeness and distribution.
    n_total   = len(raw)
    n_null    = raw[DIST_COL].isna().sum()
    n_filled  = n_total - n_null
    pct_fill  = n_filled / n_total * 100

    print(f"\n  Column found in raw data.")
    print(f"\n  Row completeness:")
    print(f"    Total rows        : {n_total:,}")
    print(f"    Rows with value   : {n_filled:,}  ({pct_fill:.1f}%)")
    print(f"    Rows null/missing : {n_null:,}  ({100 - pct_fill:.1f}%)")

    filled = raw[DIST_COL].dropna()
    print(f"\n  Distribution of CLOSE_DEF_DIST (in feet):")
    print(f"    Min    : {filled.min():.2f} ft")
    print(f"    Max    : {filled.max():.2f} ft")
    print(f"    Mean   : {filled.mean():.2f} ft")
    print(f"    Median : {filled.median():.2f} ft")
    print(f"    Std    : {filled.std():.2f} ft")

    # Show how many shots fall into each proposed contest bucket.
    # Buckets: 0-2 ft (heavily contested), 2-4 ft (contested),
    #          4-6 ft (open), 6+ ft (wide open)
    bins   = [0, 2, 4, 6, float("inf")]
    labels = [
        "Heavily Contested (0-2 ft)",
        "Contested         (2-4 ft)",
        "Open              (4-6 ft)",
        "Wide Open          (6+ ft)",
    ]
    buckets = pd.cut(filled, bins=bins, labels=labels, right=False)

    print(f"\n  Shot counts by proposed contest bucket:")
    for label, count in buckets.value_counts().sort_index().items():
        pct = count / n_filled * 100
        print(f"    {label} : {count:>7,}  ({pct:.1f}%)")

CHECK 1 — CLOSE_DEF_DIST column

  COLUMN NOT FOUND: 'CLOSE_DEF_DIST' is not in shots_2025_26.csv

  The current raw file has 25 columns:
    GRID_TYPE
    GAME_ID
    GAME_EVENT_ID
    PLAYER_ID
    PLAYER_NAME
    TEAM_ID
    TEAM_NAME
    PERIOD
    MINUTES_REMAINING
    SECONDS_REMAINING
    EVENT_TYPE
    ACTION_TYPE
    SHOT_TYPE
    SHOT_ZONE_BASIC
    SHOT_ZONE_AREA
    SHOT_ZONE_RANGE
    SHOT_DISTANCE
    LOC_X
    LOC_Y
    SHOT_ATTEMPTED_FLAG
    SHOT_MADE_FLAG
    GAME_DATE
    HTM
    VTM
    NAME

  WHY IT IS MISSING:
  Notebook 01 fetches with context_measure_simple='FGA'.
  That returns shot coordinates + outcome, but NOT defender data.
  To get CLOSE_DEF_DIST, the fetch must use context_measure_simple='DFGA'.
  The raw data needs to be re-collected before the contest-level
  breakdown can be implemented.


In [13]:
# ── Check 2: 300-shot minimum filter ─────────────────────────────────────
# Any player with fewer than 300 total field goal attempts in the 2025-26
# season will be excluded from the PPS analysis. This ensures every player
# in the dataset has a large enough sample for zone-level conclusions.
print("=" * 60)
print("CHECK 2 — 300-Shot Minimum Filter")
print("=" * 60)

MIN_FGA = 300

# Count total shot attempts per player across all games and zones
shots_per_player = (
    raw.groupby("NAME")
    .size()
    .rename("FGA")
    .sort_values(ascending=False)
)

total_players = len(shots_per_player)
qualifying    = (shots_per_player >= MIN_FGA).sum()
excluded      = total_players - qualifying

print(f"\n  Minimum FGA threshold        : {MIN_FGA}")
print(f"  Total players in raw data    : {total_players:,}")
print(f"  Players with >= {MIN_FGA} FGA   : {qualifying:,}  (qualify for analysis)")
print(f"  Players with  < {MIN_FGA} FGA   : {excluded:,}  (excluded — insufficient sample)")

print(f"\n  Shot count distribution across all {total_players} players:")
print(f"    Min    : {shots_per_player.min():.0f}  ({shots_per_player.idxmin()})")
print(f"    Median : {shots_per_player.median():.0f}")
print(f"    Mean   : {shots_per_player.mean():.0f}")
print(f"    Max    : {shots_per_player.max():.0f}  ({shots_per_player.idxmax()})")

print(f"\n  Top 10 qualifying players by FGA:")
print(f"  {'Player':<30} {'FGA':>6}")
print(f"  {'-'*30} {'------':>6}")
for name, fga in shots_per_player.head(10).items():
    print(f"  {name:<30} {fga:>6,}")

print(f"\n  Bottom 10 qualifying players (just above the {MIN_FGA}-shot cutoff):")
qualifying_players = shots_per_player[shots_per_player >= MIN_FGA]
print(f"  {'Player':<30} {'FGA':>6}")
print(f"  {'-'*30} {'------':>6}")
for name, fga in qualifying_players.tail(10).items():
    print(f"  {name:<30} {fga:>6,}")

CHECK 2 — 300-Shot Minimum Filter

  Minimum FGA threshold        : 300
  Total players in raw data    : 582
  Players with >= 300 FGA   : 284  (qualify for analysis)
  Players with  < 300 FGA   : 298  (excluded — insufficient sample)

  Shot count distribution across all 582 players:
    Min    : 1  (Tosan Evbuomwan)
    Median : 292
    Mean   : 377
    Max    : 1543  (Jaylen Brown)

  Top 10 qualifying players by FGA:
  Player                            FGA
  ------------------------------ ------
  Jaylen Brown                    1,543
  Tyrese Maxey                    1,501
  Jalen Brunson                   1,475
  Luka Dončić                     1,457
  Donovan Mitchell                1,403
  Kevin Durant                    1,376
  Jamal Murray                    1,360
  Shai Gilgeous-Alexander         1,321
  Brandon Ingram                  1,288
  Kawhi Leonard                   1,258

  Bottom 10 qualifying players (just above the 300-shot cutoff):
  Player                     

In [14]:
import pandas as pd
df = pd.read_csv('../data/raw/shots_2025_26.csv')
print(df.columns.tolist())

['GRID_TYPE', 'GAME_ID', 'GAME_EVENT_ID', 'PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_NAME', 'PERIOD', 'MINUTES_REMAINING', 'SECONDS_REMAINING', 'EVENT_TYPE', 'ACTION_TYPE', 'SHOT_TYPE', 'SHOT_ZONE_BASIC', 'SHOT_ZONE_AREA', 'SHOT_ZONE_RANGE', 'SHOT_DISTANCE', 'LOC_X', 'LOC_Y', 'SHOT_ATTEMPTED_FLAG', 'SHOT_MADE_FLAG', 'GAME_DATE', 'HTM', 'VTM', 'NAME']


In [3]:
import time
import random
from nba_api.stats.endpoints import ShotChartDetail, PlayerDashPtShots

# Stephen Curry's NBA player ID
CURRY_ID = 1629029
SEASON   = "2025-26"

# ── Part 1: Test ShotChartDetail with multiple context_measure_simple values ──
# The current notebook uses context_measure_simple="FGA".
# We test every documented valid value to confirm whether any of them
# returns a CLOSE_DEF_DIST column at the individual shot level.

print("=" * 65)
print("PART 1 — ShotChartDetail: searching for CLOSE_DEF_DIST")
print("=" * 65)

context_measures = ["FGA", "FGM", "PTS", "FG3A"]

for ctx in context_measures:
    try:
        sc = ShotChartDetail(
            team_id=0,
            player_id=CURRY_ID,
            season_nullable=SEASON,
            season_type_all_star="Regular Season",
            context_measure_simple=ctx,
        )
        # ShotChartDetail returns two result sets:
        #   [0] = shot-by-shot rows
        #   [1] = pre-computed league zone averages (fewer rows, different columns)
        result_sets = sc.get_data_frames()

        for idx, df in enumerate(result_sets):
            has_dist = "CLOSE_DEF_DIST" in df.columns
            print(
                f"  context={ctx:<6}  result_set[{idx}]  "
                f"rows={len(df):>4}  cols={len(df.columns):>3}  "
                f"CLOSE_DEF_DIST={has_dist}"
            )
            if has_dist:
                print(f"    First 5 values: {df['CLOSE_DEF_DIST'].head().tolist()}")

        time.sleep(random.uniform(2.5, 4.0))   # rate-limit courtesy delay

    except Exception as e:
        print(f"  context={ctx:<6}  ERROR: {type(e).__name__}: {e}")

print()

# Print full column list from the standard FGA result set so we see exactly
# what ShotChartDetail does return
print("Full column list from ShotChartDetail (context=FGA, result_set[0]):")
try:
    sc_fga = ShotChartDetail(
        team_id=0,
        player_id=CURRY_ID,
        season_nullable=SEASON,
        season_type_all_star="Regular Season",
        context_measure_simple="FGA",
    )
    df_fga = sc_fga.get_data_frames()[0]
    for col in df_fga.columns:
        print(f"  {col}")
    print(f"\n  Total: {len(df_fga.columns)} columns — CLOSE_DEF_DIST present: {'CLOSE_DEF_DIST' in df_fga.columns}")
except Exception as e:
    print(f"  ERROR: {type(e).__name__}: {e}")

time.sleep(random.uniform(2.5, 4.0))

# ── Part 2: Test PlayerDashPtShots — the tracking endpoint that has ───────
# defender-distance data, but in pre-aggregated bucketed form (not per-shot)
print()
print("=" * 65)
print("PART 2 — PlayerDashPtShots: CLOSE_DEF_DIST_RANGE (aggregated)")
print("=" * 65)

try:
    ep = PlayerDashPtShots(
        player_id=CURRY_ID,
        team_id=0,
        season=SEASON,
        season_type_all_star="Regular Season",
    )
    dfs = ep.get_data_frames()
    print(f"\n  PlayerDashPtShots returned {len(dfs)} result sets:")

    for i, df in enumerate(dfs):
        # Check which result set has the distance range
        dist_col = "CLOSE_DEF_DIST_RANGE" if "CLOSE_DEF_DIST_RANGE" in df.columns else None
        marker = "  <-- DISTANCE DATA" if dist_col else ""
        print(f"  [{i}] {df.shape[0]} rows x {df.shape[1]} cols{marker}")

    # Print the distance-range result set in full
    # (result sets [4] and [5] both have CLOSE_DEF_DIST_RANGE — 2PT and 3PT splits)
    for i, df in enumerate(dfs):
        if "CLOSE_DEF_DIST_RANGE" in df.columns:
            print(f"\n  Result set [{i}] — full contents (CLOSE_DEF_DIST_RANGE):")
            display_cols = ["CLOSE_DEF_DIST_RANGE", "FGA", "FGM", "FG_PCT", "EFG_PCT",
                            "FG2A", "FG2M", "FG2_PCT", "FG3A", "FG3M", "FG3_PCT"]
            available = [c for c in display_cols if c in df.columns]
            print(df[available].to_string(index=False))

except Exception as e:
    print(f"\n  ERROR: {type(e).__name__}: {e}")

# ── Summary ────────────────────────────────────────────────────────────────
print()
print("=" * 65)
print("SUMMARY")
print("=" * 65)
print("""
  ShotChartDetail (any context_measure_simple):
    Returns per-shot rows, but CLOSE_DEF_DIST is NOT available.
    The 24-column schema contains location and outcome only.

  PlayerDashPtShots (result sets [4] and [5]):
    Returns CLOSE_DEF_DIST_RANGE — pre-bucketed aggregates.
    Each row = one distance bucket (e.g. '0-2 Feet - Very Tight').
    Each row has FGA, FGM, FG_PCT, EFG_PCT for that bucket.
    This is NOT per-shot data — it is a summary table per player.

  IMPLICATION:
    Per-shot CLOSE_DEF_DIST is not exposed by the NBA stats API.
    The closest available data is CLOSE_DEF_DIST_RANGE from
    PlayerDashPtShots, which gives bucketed efficiency breakdowns
    (FGA/EFG% per distance range) at the player level — not the
    shot level. This is enough to show how a player performs under
    different contest conditions, but it cannot be joined onto
    individual shot rows in shots_2025_26.csv.
""")

PART 1 — ShotChartDetail: searching for CLOSE_DEF_DIST
  context=FGA     result_set[0]  rows=1457  cols= 24  CLOSE_DEF_DIST=False
  context=FGA     result_set[1]  rows=  20  cols=  7  CLOSE_DEF_DIST=False
  context=FGM     result_set[0]  rows=1457  cols= 24  CLOSE_DEF_DIST=False
  context=FGM     result_set[1]  rows=  20  cols=  7  CLOSE_DEF_DIST=False
  context=PTS     result_set[0]  rows= 693  cols= 24  CLOSE_DEF_DIST=False
  context=PTS     result_set[1]  rows=  20  cols=  7  CLOSE_DEF_DIST=False
  context=FG3A    result_set[0]  rows= 694  cols= 24  CLOSE_DEF_DIST=False
  context=FG3A    result_set[1]  rows=  10  cols=  7  CLOSE_DEF_DIST=False

Full column list from ShotChartDetail (context=FGA, result_set[0]):
  GRID_TYPE
  GAME_ID
  GAME_EVENT_ID
  PLAYER_ID
  PLAYER_NAME
  TEAM_ID
  TEAM_NAME
  PERIOD
  MINUTES_REMAINING
  SECONDS_REMAINING
  EVENT_TYPE
  ACTION_TYPE
  SHOT_TYPE
  SHOT_ZONE_BASIC
  SHOT_ZONE_AREA
  SHOT_ZONE_RANGE
  SHOT_DISTANCE
  LOC_X
  LOC_Y
  SHOT_ATTEMPTED_